In [26]:
import requests 
import pandas as pd 
import json
from dateutil import parser

In [3]:
API_KEY = "YOUR_OANDA_API_KEY"
ACCOUNT_ID = "YOUR_OANDA_ACCOUNT_ID"
OANDA_URL = "https://api-fxpractice.oanda.com/v3"

In [4]:
session = requests.session()

In [5]:
session.headers.update({
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
})

In [6]:
params = dict(
    count = 10,
    granularuty = "H1",
    price = "MBA"
)

In [7]:
url = f"{OANDA_URL}/accounts/{ACCOUNT_ID}/instruments/"

In [8]:
response = session.get(url, params=params, data=None, headers=None)

In [9]:
response.status_code

200

In [10]:
data = response.json()

In [11]:
instruments_list = data['instruments']

In [12]:
len(instruments_list)

123

In [13]:
instruments_list[0].keys()

dict_keys(['name', 'type', 'displayName', 'pipLocation', 'displayPrecision', 'tradeUnitsPrecision', 'minimumTradeSize', 'maximumTrailingStopDistance', 'minimumTrailingStopDistance', 'maximumPositionSize', 'maximumOrderUnits', 'marginRate', 'guaranteedStopLossOrderMode', 'tags', 'financing'])

In [14]:
key_i= ['name', 'type', 'displayName', 'pipLocation', 'displayPrecision', 'tradeUnitsPrecision','marginRate']

In [15]:
instrument_dict = {}

for i in instruments_list:
    key = (i['name'])
    instrument_dict[key] = {k: i[k] for k in key_i }

In [16]:
instrument_dict['USD_CAD']

{'name': 'USD_CAD',
 'type': 'CURRENCY',
 'displayName': 'USD/CAD',
 'pipLocation': -4,
 'displayPrecision': 5,
 'tradeUnitsPrecision': 0,
 'marginRate': '0.02'}

In [17]:
with open("../data/instrument.json", "w") as f:
    f.write(json.dumps(instrument_dict, indent=2))

In [41]:
def fetch_candles(pair_name, count=10, granularity="H1"):
    url = f"{OANDA_URL}/instruments/{pair_name}/candles"
    params = {
        "count": count,
        "granularity": granularity,
        "price": "MBA"
    }
    headers = {
        "Authorization": f"Bearer {API_KEY}"  # Replace with your actual token
    }
    response = session.get(url, params=params, headers=headers)
    data = response.json()

    if response.status_code == 200:
        data = data.get("candles", [])  # Use an empty list if 'candles' is not present
    else:
        print("Error:", response.status_code, data.get("errorMessage", "Unknown error"))

    return response.status_code, data
       

def get_candels_df(data):
    if len(data) == 0:
        return pd.DataFrame()
    
    prices = ['mid', 'bid', 'ask']
    ohlc = ['o', 'h', 'l', 'c']

    final_data = []
    for candle in data:
        new_dict = {}
        new_dict['time'] = parser.parse(candle['time'])
        new_dict['volume'] = candle['volume']
        for p in prices:
            for o in ohlc:
                new_dict[f"{p}_{o}"] = float(candle[p][o])
        final_data.append(new_dict)
    df = pd.DataFrame.from_dict(final_data)
    return df

def creat_data_file(pair_name, count=10, granularity="H1"):
    code, data = fetch_candles(pair_name, count, granularity)
    if code != 200:
        print("faild", pair_name, data)
        return
    if len(data) == 0:
        print("no candles", pair_name)
    candles_df = get_candels_df(data)  
    candles_df.to_pickle(f"../data/{pair_name}_{granularity}.pkl")  
    print(f"{pair_name}_{granularity}_{candles_df.shape[0]} {candles_df.time.min()} {candles_df.time.max()}")



In [39]:
code, data = fetch_candles("EUR_USD", count=10, granularity="H4")
candles_df = get_candels_df(data)

In [42]:
creat_data_file("EUR_USD", count=10, granularity="H4")

EUR_USD_H4_10 2024-11-05 06:00:00+00:00 2024-11-06 18:00:00+00:00


In [43]:
our_curr = ['EUR', 'USD', 'GBP', 'JPY', 'CHF', 'NZD', 'CAD', 'AUD']

In [44]:
for p1 in our_curr:
    for p2 in our_curr:
        pr = f"{p1}_{p2}"
        if pr in instrument_dict:
            for g in["H1", "H4"]:
               creat_data_file(pr, count=4001, granularity=g)

EUR_USD_H1_4001 2024-03-19 00:00:00+00:00 2024-11-06 19:00:00+00:00
EUR_USD_H4_4001 2022-04-14 01:00:00+00:00 2024-11-06 18:00:00+00:00
EUR_GBP_H1_4001 2024-03-19 00:00:00+00:00 2024-11-06 19:00:00+00:00
EUR_GBP_H4_4001 2022-04-14 01:00:00+00:00 2024-11-06 18:00:00+00:00
EUR_JPY_H1_4001 2024-03-19 00:00:00+00:00 2024-11-06 19:00:00+00:00
EUR_JPY_H4_4001 2022-04-13 17:00:00+00:00 2024-11-06 18:00:00+00:00
EUR_CHF_H1_4001 2024-03-19 00:00:00+00:00 2024-11-06 19:00:00+00:00
EUR_CHF_H4_4001 2022-04-14 01:00:00+00:00 2024-11-06 18:00:00+00:00
EUR_NZD_H1_4001 2024-03-19 00:00:00+00:00 2024-11-06 19:00:00+00:00
EUR_NZD_H4_4001 2022-04-14 09:00:00+00:00 2024-11-06 18:00:00+00:00
EUR_CAD_H1_4001 2024-03-19 00:00:00+00:00 2024-11-06 19:00:00+00:00
EUR_CAD_H4_4001 2022-04-14 01:00:00+00:00 2024-11-06 18:00:00+00:00
EUR_AUD_H1_4001 2024-03-19 00:00:00+00:00 2024-11-06 19:00:00+00:00
EUR_AUD_H4_4001 2022-04-14 01:00:00+00:00 2024-11-06 18:00:00+00:00
USD_JPY_H1_4001 2024-03-19 00:00:00+00:00 2024-1